# 02 · Training Runs and Results

The training itself runs via `train_arch.py` (hours per model); this notebook is the record.
It reproduces the full logs, the results tables of Section 5, and Figures 6, 7 and 9.

In [ ]:
import sys, os, re
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../paper'))
%matplotlib inline
from pathlib import Path
import pandas as pd
LOGS = Path('../logs')
print('logs found:', sorted(p.name for p in LOGS.glob('*.log')))

## Training configuration

Every architecture shares `train_merged.py`'s manifest, transforms and seed-42 split, so all
runs see byte-identical data. Only the model varies.

In [ ]:
from train_arch import ARCHS
for name, spec in ARCHS.items():
    print(f'{name:14s} default lr {spec["lr"]}')
print('''
Shared: AdamW (wd 1e-4), cross-entropy w/ label smoothing 0.1,
cosine anneal to 1e-6 over 8 epochs, batch 32, grad-clip 1.0,
backbone lr = 0.1 x base for pretrained models.''')

## Complete training logs

In [ ]:
RUNS = [('cnn', '300/class'), ('resnet18', '300/class'), ('extended_vit', '300/class'),
        ('vit', '300/class'), ('extended_vit_imagenet', '300/class, ImageNet init'),
        ('cnn_full', 'full corpus')]
for stem, cond in RUNS:
    p = LOGS / f'{stem}.log'
    print('=' * 78)
    print(f'{stem}  ({cond})')
    print('=' * 78)
    if not p.exists():
        print('(not run)\n'); continue
    for line in p.read_text(errors='replace').splitlines():
        if any(k in line for k in ('Epoch [', 'Best Val', 'Test Acc', 'Total params',
                                   'Warm-started', 'Device', 'Train ', 'images,')):
            print(line)
    print()

## Results tables

In [ ]:
for csv, title in [('../results_lpc300.csv', 'Equal budget (300 per class)'),
                   ('../results_lpc300_imagenet.csv', 'Equal budget, ImageNet-init ablation'),
                   ('../results_full.csv', 'Full corpus')]:
    print(f'\n### {title}')
    try:
        print(pd.read_csv(csv).to_string(index=False))
    except FileNotFoundError:
        print('(evaluate.py has not produced this yet)')

## Figures 6, 7 and 9

In [ ]:
from figures import fig6_training_curves, fig7_results, fig9_efficiency
for fn in (fig6_training_curves, fig7_results, fig9_efficiency):
    fn(); print(fn.__name__, 'written')

In [ ]:
from IPython.display import Image, display
for f in ('fig6_training_curves', 'fig7_results', 'fig9_efficiency'):
    display(Image(f'../paper/{f}.png'))

## Confusion matrices

In [ ]:
from IPython.display import Image, display
import glob
found = sorted(glob.glob('../paper/confusion_*.png'))
print(found if found else '(produced by evaluate.py --confusion)')
for f in found:
    display(Image(f))